In [1]:
import io
import os
import requests
import pandas as pd
from google.cloud import storage

In [2]:

# services = ['fhv','green','yellow']
init_url = 'https://github.com/DataTalksClub/nyc-tlc-data/releases/download/'

In [69]:
i = 6
year = "2019"
service = "green"

In [70]:
# sets the month part of the file_name string
month = '0'+str(i+1)
month = month[-2:]

# csv file_name
file_name = f"{service}_tripdata_{year}-{month}.csv.gz"

# download it using requests via a pandas df
request_url = f"{init_url}{service}/{file_name}"
r = requests.get(request_url)
open(file_name, 'wb').write(r.content)
print(f"Local: {file_name}")

Local: green_tripdata_2019-07.csv.gz


In [71]:
!ls

Define_schema.ipynb           green_tripdata_2019-07.csv.gz
README.md                     taxi_rides_ny
assets                        terraform
dbt.md                        web_to_gcs.py
environment.yaml


In [73]:
df = pd.read_csv(file_name, compression='gzip',low_memory=False)

In [74]:
df.head()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge
0,2.0,2019-07-01 00:00:48,2019-07-01 00:04:39,N,1.0,17,17,1.0,0.58,4.5,0.5,0.5,0.00,0.0,NaN,0.3,5.80,1.0,1.0,0.0
1,2.0,2019-07-01 00:23:36,2019-07-01 00:29:50,N,1.0,255,256,1.0,0.95,6.0,0.5,0.5,1.46,0.0,NaN,0.3,8.76,1.0,1.0,0.0
2,2.0,2019-07-01 00:09:48,2019-07-01 00:26:09,N,1.0,75,116,2.0,3.61,14.5,0.5,0.5,2.00,0.0,NaN,0.3,17.80,1.0,1.0,0.0
3,2.0,2019-07-01 00:09:07,2019-07-01 00:23:41,N,1.0,17,89,1.0,3.59,13.5,0.5,0.5,0.00,0.0,NaN,0.3,14.80,2.0,1.0,0.0
4,2.0,2019-07-01 00:50:45,2019-07-01 01:03:51,N,1.0,65,195,1.0,2.21,10.5,0.5,0.5,0.00,0.0,NaN,0.3,11.80,2.0,1.0,0.0


In [75]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 470743 entries, 0 to 470742
Data columns (total 20 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   VendorID               433376 non-null  float64
 1   lpep_pickup_datetime   470743 non-null  object 
 2   lpep_dropoff_datetime  470743 non-null  object 
 3   store_and_fwd_flag     433376 non-null  object 
 4   RatecodeID             433376 non-null  float64
 5   PULocationID           470743 non-null  int64  
 6   DOLocationID           470743 non-null  int64  
 7   passenger_count        433376 non-null  float64
 8   trip_distance          470743 non-null  float64
 9   fare_amount            470743 non-null  float64
 10  extra                  470743 non-null  float64
 11  mta_tax                470743 non-null  float64
 12  tip_amount             470743 non-null  float64
 13  tolls_amount           470743 non-null  float64
 14  ehail_fee              272 non-null 

In [76]:
# cast time data to timestamp
time_cols = df.filter(regex='datetime').columns
print(list(time_cols))
for column in time_cols:
    df[column] = pd.to_datetime(df[column])

['lpep_pickup_datetime', 'lpep_dropoff_datetime']


In [77]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 470743 entries, 0 to 470742
Data columns (total 20 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   VendorID               433376 non-null  float64       
 1   lpep_pickup_datetime   470743 non-null  datetime64[ns]
 2   lpep_dropoff_datetime  470743 non-null  datetime64[ns]
 3   store_and_fwd_flag     433376 non-null  object        
 4   RatecodeID             433376 non-null  float64       
 5   PULocationID           470743 non-null  int64         
 6   DOLocationID           470743 non-null  int64         
 7   passenger_count        433376 non-null  float64       
 8   trip_distance          470743 non-null  float64       
 9   fare_amount            470743 non-null  float64       
 10  extra                  470743 non-null  float64       
 11  mta_tax                470743 non-null  float64       
 12  tip_amount             470743 non-null  floa

In [88]:
# also cast IDs and types to integer
id_cols = df.filter(regex='ID|type').columns
print(list(id_cols))
for column in id_cols:
    print(column)
    df[column] = df[column].astype('Int64')

['VendorID', 'RatecodeID', 'PULocationID', 'DOLocationID', 'payment_type', 'trip_type']
VendorID
RatecodeID
PULocationID
DOLocationID
payment_type
trip_type


In [85]:
df.VendorID

0         2.0
1         2.0
2         2.0
3         2.0
4         2.0
         ... 
470738    NaN
470739    NaN
470740    NaN
470741    NaN
470742    NaN
Name: VendorID, Length: 470743, dtype: float64

In [ ]:
df.info()

In [44]:
df.store_and_fwd_flag.head()

0    N
1    N
2    N
3    N
4    N
Name: store_and_fwd_flag, dtype: object

In [60]:
def fix_types(df):
    # cast time data to timestamp
    for column in df.filter(regex='datetime').columns:
        df[column] = pd.to_datetime(df[column])
    # also cast IDs and types to integer
    for column in df.filter(regex='ID|type').columns:
        df[column] = df[column].astype('int') 
    return df

In [63]:
# read it back into a parquet file
df = pd.read_csv(file_name, compression='gzip')
df = fix_types(df)
file_name = file_name.replace('.csv.gz', '.parquet')
df.to_parquet(file_name, engine='pyarrow')
print(f"Parquet: {file_name}")

Parquet: green_tripdata_2019-01.parquet


In [64]:
!ls

Define_schema.ipynb            green_tripdata_2019-01.parquet
README.md                      green_tripdata_2019-02.csv.gz
assets                         green_tripdata_2019-02.parquet
dbt.md                         taxi_rides_ny
environment.yaml               web_to_gcs.py
green_tripdata_2019-01.csv.gz


In [65]:
# check parquet schema
pq = pd.read_parquet(file_name)

In [66]:
pq.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630918 entries, 0 to 630917
Data columns (total 20 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   VendorID               630918 non-null  int64         
 1   lpep_pickup_datetime   630918 non-null  datetime64[ns]
 2   lpep_dropoff_datetime  630918 non-null  datetime64[ns]
 3   store_and_fwd_flag     630918 non-null  object        
 4   RatecodeID             630918 non-null  int64         
 5   PULocationID           630918 non-null  int64         
 6   DOLocationID           630918 non-null  int64         
 7   passenger_count        630918 non-null  int64         
 8   trip_distance          630918 non-null  float64       
 9   fare_amount            630918 non-null  float64       
 10  extra                  630918 non-null  float64       
 11  mta_tax                630918 non-null  float64       
 12  tip_amount             630918 non-null  floa

In [68]:
?os.remove

Signature: os.remove(path, *, dir_fd=None)
Docstring:
Remove a file (same as unlink()).

If dir_fd is not None, it should be a file descriptor open to a directory,
  and path should be relative; path will then be relative to that directory.
dir_fd may not be implemented on your platform.
  If it is unavailable, using it will raise a NotImplementedError.
Type:      builtin_function_or_method

In [ ]:
o